<a href="https://colab.research.google.com/github/ehsanre1376/YouTube-DownLoader-To-Colab/blob/main/YouTubeToGoogleDrive5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Installation and Setup ---
# Update yt-dlp first to get latest features/fixes
!pip install --upgrade yt-dlp -q
!pip install ipywidgets -q
!sudo apt-get update -qq && sudo apt-get install ffmpeg -qq

import os
import re
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from yt_dlp import YoutubeDL
from yt_dlp.utils import DownloadError # Correct import
from yt_dlp.postprocessor import FFmpegExtractAudioPP # For MP3 conversion
from google.colab import drive
import logging
import time
import math

# --- Configuration ---
# Set logging to INFO to see yt-dlp's own info messages (like cookie usage)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s:%(name)s:%(message)s')
logger = logging.getLogger(__name__)

# --- Constants ---
DEFAULT_DRIVE_FOLDER = "YT_Downloads"
DEFAULT_COOKIES_FILENAME = "www.youtube.com_cookies.txtt" # Just the filename

# Mount Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
    BASE_DRIVE_PATH = '/content/drive/MyDrive/'
    drive_mounted = True
    logger.info("Google Drive mounted successfully.")
    # Construct default cookies path based on Drive mount
    DEFAULT_COOKIES_PATH = os.path.join(BASE_DRIVE_PATH, DEFAULT_DRIVE_FOLDER, DEFAULT_COOKIES_FILENAME)
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
    print("Downloads will be saved to Colab's temporary storage (/content/downloads) instead.")
    BASE_DRIVE_PATH = '/content/downloads/' # Fallback path
    drive_mounted = False
    os.makedirs(BASE_DRIVE_PATH, exist_ok=True)
    logger.warning("Google Drive mount failed. Using temporary storage.")
    DEFAULT_COOKIES_PATH = f"/content/{DEFAULT_COOKIES_FILENAME}" # Suggest temporary location

# --- Global State ---
app_state = {
    'status': 'idle', # idle, waiting_for_range, downloading
    'url': None,
    'initial_info': None,
    'output_base_path': None,
    'media_type': None,
    'playlist_total_items': 0,
    'cookies_path': None, # Holds the validated path to the REQUIRED cookies file
    # Store user choices for easier passing
    'download_format': 'video', # video, audio
    'download_thumbnail': False,
    'download_metadata': False,
    'playlist_delay': 0,
}

# --- Helper Functions --- (sanitize_name, get_base_output_directory - Unchanged)
def sanitize_name(name):
    name = str(name) if name is not None else ""
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = re.sub(r'\.{2,}', '.', name)
    name = name.strip(' .')
    if not name: name = "untitled"
    return name

def get_base_output_directory(info, output_base, media_type):
    title = "Unknown"
    if media_type == 'playlist':
        title = info.get('playlist_title') or info.get('title', 'Unknown Playlist')
    else: # video
        title = info.get('title', 'Unknown Video')
    sanitized_title = sanitize_name(title)
    return os.path.join(output_base, sanitized_title)

# --- Main Download Logic --- (Handles new options)
def download_media(url, media_type, output_base, progress_callback, initial_info,
                   cookies_path, user_options, playlist_range=None):
    """Downloads media using yt-dlp with user-specified options."""

    if not cookies_path or not os.path.exists(cookies_path):
         raise ValueError("Invalid or missing cookies file path passed to download_media.")

    final_base_dir = get_base_output_directory(initial_info, output_base, media_type)
    os.makedirs(final_base_dir, exist_ok=True)
    logger.info(f"Ensured output directory exists: {final_base_dir}")

    # --- Determine Output Template Base ---
    if media_type == 'playlist':
         template_base = os.path.join(final_base_dir, '%(playlist_autonumber)04d-%(title)s')
    else: # Single Video
         video_subfolder_name = sanitize_name(initial_info.get('title', 'Unknown Video'))
         video_subfolder_path = os.path.join(final_base_dir, video_subfolder_name)
         os.makedirs(video_subfolder_path, exist_ok=True)
         template_base = os.path.join(video_subfolder_path, '%(title)s')
    logger.info(f"Using template base: {template_base}")

    # --- Base yt-dlp Options ---
    ydl_opts = {
        'outtmpl': template_base,
        'writesubtitles': True,
        'subtitleslangs': ['en', 'fa'],
        'writeautomaticsub': True,
        'subtitlesformat': 'srt',
        'embedsubtitles': False, # Keep subtitles separate
        'quiet': True, 'no_warnings': True,
        'progress_hooks': [progress_callback],
        'postprocessor_args': {'ffmpeg': ['-loglevel', 'error']},
        'fragment_retries': 10,
        'retry_sleep_functions': {'http': lambda n: min(n * 5, 30),'fragment': lambda n: min(n * 5, 30)},
        'ignoreerrors': True,
        'cookiefile': cookies_path,
        'logger': logger, # Pass our logger to yt-dlp
    }

    # --- Apply User Format Choice ---
    postprocessors = []
    if user_options['download_format'] == 'audio':
        ydl_opts['format'] = 'bestaudio/best'
        ydl_opts['extract_audio'] = True # Ensure audio extraction is triggered
        postprocessors.append({
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192', # Quality for MP3 VBR
        })
        logger.info("Configured for Audio Only (MP3) download.")
    else: # Video (default)
        ydl_opts['format'] = 'bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best[height<=1080][ext=mp4]/best[height<=1080]'
        ydl_opts['merge_output_format'] = 'mp4'
        postprocessors.append({'key': 'FFmpegMetadata', 'add_metadata': True})
        postprocessors.append({'key': 'FFmpegVideoConvertor','preferedformat': 'mp4'})
        logger.info("Configured for Video + Audio (MP4) download.")

    ydl_opts['postprocessors'] = postprocessors

    # --- Apply Optional Settings ---
    if user_options['download_thumbnail']:
        ydl_opts['writethumbnail'] = True
        logger.info("Thumbnail download enabled.")
    if user_options['download_metadata']:
        ydl_opts['writeinfojson'] = True # Gets the comprehensive .info.json
        # ydl_opts['writedescription'] = True # Optionally get .description file
        logger.info("Metadata (.info.json) download enabled.")
    if media_type == 'playlist' and user_options['playlist_delay'] > 0:
        ydl_opts['sleep_interval'] = user_options['playlist_delay']
        logger.info(f"Playlist item delay set to: {user_options['playlist_delay']}s")

    # Add Playlist Range Option
    if media_type == 'playlist' and playlist_range:
        ydl_opts['playlist_items'] = playlist_range
        logger.info(f"Applying playlist range: {playlist_range}")

    # --- Execute Download ---
    download_error = None; download_interrupted = False
    try:
        logger.info(f"Initializing YoutubeDL for download...") # Options logged by yt-dlp itself now
        with YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
            logger.info(f"yt-dlp download process completed for URL: {url}")
    except Exception as e:
        logger.error(f"yt-dlp download failed/stopped: {e}", exc_info=True)
        download_error = str(e)
        if isinstance(e, KeyboardInterrupt): download_interrupted = True; logger.warning("Download interrupted by user.")

    return final_base_dir, download_error, download_interrupted


# --- GUI Components ---
url_widget = widgets.Text(placeholder='Enter YouTube Video or Playlist URL', layout={'width': '95%'})
drive_folder_widget = widgets.Text(value=DEFAULT_DRIVE_FOLDER, description='Drive Folder:', placeholder='Subfolder in MyDrive', layout={'width': '95%'})
cookies_path_widget = widgets.Text(
    value=DEFAULT_COOKIES_PATH, # Pre-fill likely path
    placeholder='e.g., /content/drive/MyDrive/YT_Downloads/www.youtube.com_cookies.txt',
    description='Cookies File Path:', layout={'width': '95%'}
)
cookies_help_html = widgets.HTML(value="<p><small><b>REQUIRED:</b> Export cookies from your browser (use extension for youtube.com) & upload the file. Paste the <b>full path</b> here. <b style='color:red;'>Treat like password.</b></small></p>")

# --- New Options Widgets ---
download_type_widget = widgets.Dropdown(options=[('Video + Audio (MP4)', 'video'), ('Audio Only (MP3)', 'audio')], value='video', description='Download Type:')
thumbnail_widget = widgets.Checkbox(value=False, description='Download Thumbnail', indent=False)
metadata_widget = widgets.Checkbox(value=False, description='Download Metadata (.info.json)', indent=False)
delay_widget = widgets.IntText(value=0, description='Playlist Delay (sec):', layout={'width': '200px'}, min=0, step=1, tooltip="Seconds to wait between playlist items (helps avoid rate limits)")
options_box = widgets.VBox([
    download_type_widget,
    widgets.HBox([thumbnail_widget, metadata_widget]),
    delay_widget
])
# ---------------------------

start_item_widget = widgets.IntText(value=1, description="Start Item:", layout={'width': '150px'}, min=1)
end_item_widget = widgets.IntText(value=0, description="End Item (0=all):", layout={'width': '180px'}, min=0)
playlist_range_box = widgets.HBox([start_item_widget, end_item_widget], layout={'display': 'none'})

download_btn = widgets.Button(description='Get Info & Download', button_style='success', icon='download', tooltip='Click to start fetching info or confirm playlist range')
progress_widget = widgets.FloatProgress(value=0.0, min=0.0, max=1.0, description='Progress:', bar_style='info', orientation='horizontal', layout={'width': '95%'})
status_widget = widgets.HTML(value="<b>Status:</b> Ready")
output_widget = widgets.Output()


# --- Button Click Handler (Handles New Options) ---
def on_button_click(b):
    global app_state

    # --- Phase 2: User confirmed playlist range --- (Largely unchanged internal logic)
    if app_state['status'] == 'waiting_for_range':
        logger.info("Button clicked: Phase 2 - Confirming playlist range.")
        download_btn.disabled = True; download_btn.description = 'Starting Download...'; download_btn.button_style = 'warning'
        playlist_range_box.layout.display = 'none'

        start_item = start_item_widget.value; end_item = end_item_widget.value
        # (Range validation - unchanged) ...
        if not isinstance(start_item, int) or start_item < 1: status_widget.value = "<b style='color:red;'>Error:</b> Start item must be >= 1."; logger.error(f"Invalid start item: {start_item}"); download_btn.disabled = False; download_btn.description = 'Confirm Range & Download'; download_btn.button_style = 'info'; playlist_range_box.layout.display = 'flex'; return
        if not isinstance(end_item, int) or end_item < 0: status_widget.value = "<b style='color:red;'>Error:</b> End item must be >= 0."; logger.error(f"Invalid end item: {end_item}"); download_btn.disabled = False; download_btn.description = 'Confirm Range & Download'; download_btn.button_style = 'info'; playlist_range_box.layout.display = 'flex'; return
        if end_item != 0 and end_item < start_item: status_widget.value = "<b style='color:red;'>Error:</b> End item < Start item."; logger.error(f"Invalid range: end({end_item}) < start({start_item})"); download_btn.disabled = False; download_btn.description = 'Confirm Range & Download'; download_btn.button_style = 'info'; playlist_range_box.layout.display = 'flex'; return
        logger.info(f"Playlist range confirmed: Start={start_item}, End={end_item}")

        # (Construct Playlist Range String - unchanged) ...
        playlist_range_str = None; total_items_in_range = 0; playlist_total_items = app_state['playlist_total_items']
        if start_item == 1 and end_item == 0: playlist_range_str = None; total_items_in_range = playlist_total_items
        elif end_item == 0: clamped_start = min(start_item, playlist_total_items + 1); playlist_range_str = f"{clamped_start}:"; total_items_in_range = playlist_total_items - clamped_start + 1
        else: clamped_start = min(start_item, playlist_total_items + 1); clamped_end = min(end_item, playlist_total_items)
        if clamped_end < clamped_start: total_items_in_range = 0; playlist_range_str = f"{clamped_start}:{clamped_end}"
        else: playlist_range_str = f"{clamped_start}:{clamped_end}"; total_items_in_range = clamped_end - clamped_start + 1
        if total_items_in_range < 0: total_items_in_range = 0

        logger.info(f"Calculated playlist range string: '{playlist_range_str}', Expected items: {total_items_in_range}")
        app_state['status'] = 'downloading'

        # (Define progress_hook_playlist - unchanged) ...
        last_playlist_index = 0; current_overall_item_count = 0
        def progress_hook_playlist(d):
            nonlocal last_playlist_index, current_overall_item_count
            try:
                current_index = d.get('info_dict', {}).get('playlist_index')
                if current_index is not None and current_index != last_playlist_index:
                    last_playlist_index = current_index; current_overall_item_count += 1
                    display_count = min(current_overall_item_count, total_items_in_range)
                    overall_progress = (display_count / total_items_in_range) if total_items_in_range > 0 else 0.0
                    progress_widget.value = overall_progress; progress_widget.bar_style = 'info'
                    item_title = sanitize_name(d.get('info_dict', {}).get('title', '...')); display_title = (item_title[:40] + '...') if len(item_title) > 43 else item_title
                    status_widget.value = (f"<i>Playlist Item {display_count}/{total_items_in_range}: Downloading '{display_title}'...</i>")
                elif d['status'] == 'error': logger.error(f"Item error hook: {d.get('filename')}"); status_widget.value += " <i style='color:orange;'>(item error)</i>"
                elif d['status'] == 'postprocessing':
                     item_title = sanitize_name(d.get('info_dict', {}).get('title', '...')); display_title = (item_title[:40] + '...') if len(item_title) > 43 else item_title
                     display_count = min(current_overall_item_count, total_items_in_range)
                     status_widget.value = (f"<i>Playlist Item {display_count}/{total_items_in_range}: Post-processing '{display_title}'...</i>")
            except Exception as e: logger.error(f"Error in playlist progress_hook: {e}", exc_info=True)

        # --- Execute Download (Phase 2 - Pass User Options) ---
        final_path = None; download_error_msg = None; was_interrupted = False
        try:
            # Collate user options from state
            user_options = {
                'download_format': app_state['download_format'],
                'download_thumbnail': app_state['download_thumbnail'],
                'download_metadata': app_state['download_metadata'],
                'playlist_delay': app_state['playlist_delay'],
            }
            final_path, download_error_msg, was_interrupted = download_media(
                app_state['url'], app_state['media_type'], app_state['output_base_path'],
                progress_hook_playlist, app_state['initial_info'],
                cookies_path=app_state['cookies_path'], # Required
                user_options=user_options,             # Pass collected options
                playlist_range=playlist_range_str
            )
            # (Final status reporting - unchanged) ...
            if was_interrupted: status_widget.value = "<b style='color:orange;'>Download Interrupted.</b> Partial files may exist."; progress_widget.bar_style = 'warning'; logger.warning("Playlist download interrupted.")
            elif download_error_msg and final_path and not was_interrupted: status_widget.value = (f"<b style='color:orange;'>Warning:</b> Some playlist items might have failed (check logs). Content saved to: <code>{final_path}</code>"); progress_widget.bar_style = 'warning'; progress_widget.value = 1.0; logger.warning(f"Playlist completed with errors: {download_error_msg}")
            elif download_error_msg: status_widget.value = f"<b style='color:red;'>Error:</b> {download_error_msg}"; progress_widget.bar_style = 'danger'; logger.error(f"Playlist download failed completely: {download_error_msg}")
            elif final_path: progress_widget.value = 1.0; progress_widget.bar_style = 'success'; status_widget.value = f"<b>Success!</b> Playlist items saved to: <code>{final_path}</code>"; logger.info(f"Playlist download successful. Path: {final_path}")
            else: status_widget.value = "<b style='color:red;'>Error:</b> Download process finished but final path is unknown."; progress_widget.bar_style = 'danger'; logger.error("Playlist finished, but final path unknown.")

        except Exception as e:
            logger.error(f"Critical error during playlist download phase: {e}", exc_info=True)
            status_widget.value = f"<b style='color:red;'>Critical Error:</b> {str(e)}"; progress_widget.bar_style = 'danger'
        finally:
            # (Reset state and button - unchanged) ...
            app_state['status'] = 'idle'; download_btn.disabled = False; download_btn.description = 'Get Info & Download'
            if progress_widget.bar_style == 'success': download_btn.button_style = 'success'
            elif progress_widget.bar_style == 'warning': download_btn.button_style = 'warning'
            else: download_btn.button_style = 'danger'
            logger.info("Playlist download phase finished, UI reset.")


    # --- Phase 1: Initial button click - Get Info (Handles New Options) ---
    elif app_state['status'] == 'idle':
        logger.info("Button clicked: Phase 1 - Get Info.")
        download_btn.disabled = True; download_btn.description = 'Processing...'; download_btn.button_style = 'warning'
        progress_widget.value = 0.0; progress_widget.bar_style = 'info'
        status_widget.value = "<b>Status:</b> Checking inputs..."
        playlist_range_box.layout.display = 'none'

        with output_widget: clear_output(wait=True)

        # --- Read All Inputs ---
        url = url_widget.value.strip()
        drive_subfolder = drive_folder_widget.value.strip()
        cookies_path_input = cookies_path_widget.value.strip()
        # Read new option widgets
        app_state['download_format'] = download_type_widget.value
        app_state['download_thumbnail'] = thumbnail_widget.value
        app_state['download_metadata'] = metadata_widget.value
        app_state['playlist_delay'] = delay_widget.value if delay_widget.value >= 0 else 0 # Ensure non-negative

        # --- *** MANDATORY Input Checks *** ---
        if not url: status_widget.value = "<b style='color:red;'>Error:</b> URL required."; logger.warning("Validation failed: URL empty."); app_state['status'] = 'idle'; download_btn.disabled = False; download_btn.description = 'Get Info & Download'; download_btn.button_style = 'success'; return
        if not cookies_path_input: status_widget.value = "<b style='color:red;'>Error:</b> Cookies file path required."; logger.warning("Validation failed: Cookies path empty."); app_state['status'] = 'idle'; download_btn.disabled = False; download_btn.description = 'Get Info & Download'; download_btn.button_style = 'success'; return
        if not os.path.exists(cookies_path_input): status_widget.value = f"<b style='color:red;'>Error:</b> Cookies file NOT FOUND: <code>{cookies_path_input}</code>"; logger.warning(f"Validation failed: Cookies file not found at {cookies_path_input}"); app_state['status'] = 'idle'; download_btn.disabled = False; download_btn.description = 'Get Info & Download'; download_btn.button_style = 'success'; return

        # --- Inputs Validated - Store State ---
        app_state['cookies_path'] = cookies_path_input
        logger.info(f"Inputs validated. Using cookies: {app_state['cookies_path']}")
        status_widget.value = "<b>Status:</b> Inputs OK. Fetching info..."

        # --- Construct Base Output Path ---
        sanitized_subfolder = sanitize_name(drive_subfolder) if drive_subfolder else DEFAULT_DRIVE_FOLDER
        full_output_base_path = os.path.join(BASE_DRIVE_PATH, sanitized_subfolder)
        app_state['output_base_path'] = full_output_base_path

        # (Define progress_hook_single - unchanged) ...
        def progress_hook_single(d):
            try:
                if d['status'] == 'downloading':
                    percent_str = d.get('_percent_str', '0%').strip().strip('%');
                    try: progress_widget.value = float(percent_str) / 100.0
                    except ValueError: pass
                    base_filename = os.path.basename(d.get('filename', '...')); display_filename = (base_filename[:50] + '...') if len(base_filename) > 53 else base_filename
                    status_widget.value = f"<i>Downloading: {display_filename} ({d.get('_speed_str', '?')})</i>"; progress_widget.bar_style = 'info'
                elif d['status'] == 'error': logger.error(f"Item error hook: {d.get('filename')}"); status_widget.value = f"<b style='color:orange;'>Warning:</b> Error downloading {os.path.basename(d.get('filename', 'file'))}."; progress_widget.bar_style = 'warning'
                elif d['status'] == 'finished': progress_widget.value = 1.0; status_widget.value = f"<i>Finished: {os.path.basename(d.get('filename', '...'))}. Post-processing...</i>"; progress_widget.bar_style = 'info'
                elif d['status'] == 'postprocessing': status_widget.value = f"<i>Post-processing...</i>"
            except Exception as e: logger.error(f"Error in single video progress_hook: {e}", exc_info=True)

        # --- Detect Type (Using MANDATORY Cookies) ---
        initial_info = None; media_type = None
        try:
            status_widget.value = "<i>Detecting URL type (using cookies)...</i>"
            info_opts = {
                'quiet': True, 'no_warnings': True, 'extract_flat': 'discard_in_playlist',
                'simulate': True, 'logger': logger, 'cookiefile': app_state['cookies_path'],
            }
            logger.info(f"Passing mandatory cookies file to yt-dlp for info extraction: {app_state['cookies_path']}")
            time.sleep(0.2) # UI update delay

            with YoutubeDL(info_opts) as ydl_info:
                 logger.info(f"Calling yt-dlp extract_info for URL: {url}")
                 initial_info = ydl_info.extract_info(url, download=False)
                 logger.info("extract_info call finished successfully.")
                 app_state['initial_info'] = initial_info

            # --- Process Based on Detected Type ---
            if initial_info and initial_info.get('_type') == 'playlist':
                # (Playlist detected - logic unchanged, prepares for Phase 2) ...
                media_type = 'playlist'; app_state['media_type'] = media_type
                playlist_title = sanitize_name(initial_info.get('title', 'Unknown Playlist')); playlist_count = initial_info.get('playlist_count', 0)
                if playlist_count == 0 and 'entries' in initial_info: playlist_count = len(initial_info['entries'])
                app_state['playlist_total_items'] = playlist_count
                logger.info(f"Playlist detected: '{playlist_title}', Count: {playlist_count}")
                status_widget.value = (f"<b>Playlist Detected:</b> '{playlist_title}' ({playlist_count} items).<br>Specify range and click again.")
                app_state['status'] = 'waiting_for_range'; app_state['url'] = url
                start_item_widget.value = 1; end_item_widget.value = playlist_count
                playlist_range_box.layout.display = 'flex'; download_btn.disabled = False
                download_btn.description = 'Confirm Range & Download'; download_btn.button_style = 'info'
                logger.info("UI configured for playlist range confirmation.")
                return

            elif initial_info: # Single video
                media_type = 'video'; app_state['media_type'] = media_type
                video_title = sanitize_name(initial_info.get('title', 'Unknown Video'))
                logger.info(f"Single video detected: '{video_title}'")
                # Update status based on download type
                download_desc = "Audio" if app_state['download_format'] == 'audio' else "Video"
                status_widget.value = f"<i>Single {download_desc} detected: '{video_title}'. Starting download...</i>"
                app_state['status'] = 'downloading'

                # Execute Download (Single Video - Pass User Options)
                user_options = { # Collate options for single download
                    'download_format': app_state['download_format'],
                    'download_thumbnail': app_state['download_thumbnail'],
                    'download_metadata': app_state['download_metadata'],
                    'playlist_delay': 0, # Not applicable
                }
                final_path, download_error_msg, was_interrupted = download_media(
                    url, media_type, full_output_base_path, progress_hook_single,
                    initial_info, cookies_path=app_state['cookies_path'], # Required
                    user_options=user_options
                )
                # (Final status reporting - unchanged) ...
                if was_interrupted: status_widget.value = "<b style='color:orange;'>Download Interrupted.</b>"; progress_widget.bar_style = 'warning'; logger.warning("Single download interrupted.")
                elif download_error_msg: status_widget.value = f"<b style='color:red;'>Error:</b> {download_error_msg}"; progress_widget.bar_style = 'danger'; logger.error(f"Single download failed: {download_error_msg}")
                elif final_path: progress_widget.value = 1.0; progress_widget.bar_style = 'success'; status_widget.value = f"<b>Success!</b> Content saved to: <code>{final_path}</code>"; logger.info(f"Single download successful. Path: {final_path}")
                else: status_widget.value = "<b style='color:red;'>Error:</b> Download finished but final path unknown."; progress_widget.bar_style = 'danger'; logger.error("Single download finished, but final path unknown.")

            else:
                 raise ValueError("Failed to extract information, even with cookies.")

        except DownloadError as e: # Catch yt-dlp specific errors
             # Refined error message for auth failure with mandatory cookies
             if 'confirm you' in str(e) or 'sign in' in str(e).lower() or 'unavailable' in str(e).lower() or 'age-restricted' in str(e).lower() or 'verify your identity' in str(e).lower():
                 error_msg = "<b>Error:</b> Authentication failed or video unavailable/restricted. Ensure cookies are valid/up-to-date (re-export/upload if needed)."
                 status_widget.value = f"<span style='color:red;'>{error_msg}</span>"
                 logger.error(f"Authentication/Restriction Error from yt-dlp (cookies REQUIRED): {e}")
             else: # Other yt-dlp download errors
                 error_msg = f"<b>Download Error:</b> {str(e)}"
                 status_widget.value = f"<span style='color:red;'>{error_msg}</span>"
                 logger.error(f"Unhandled DownloadError during info extraction: {e}", exc_info=True)
             progress_widget.bar_style = 'danger'; app_state['status'] = 'idle'

        except Exception as e: # Catch any other unexpected errors
            error_msg = f"<b>Error:</b> {str(e)}"
            status_widget.value = f"<span style='color:red;'>{error_msg}</span>"
            logger.error(f"Unexpected error during initial processing or single download: {e}", exc_info=True)
            progress_widget.bar_style = 'danger'; app_state['status'] = 'idle'

        finally:
             # (Reset button logic - unchanged) ...
             if app_state['status'] != 'waiting_for_range':
                app_state['status'] = 'idle'
                download_btn.disabled = False; download_btn.description = 'Get Info & Download'
                if progress_widget.bar_style == 'success': download_btn.button_style = 'success'
                elif progress_widget.bar_style == 'warning': download_btn.button_style = 'warning'
                else: download_btn.button_style = 'danger'
                logger.info("Phase 1 finished or failed, UI reset.")


# --- Display GUI --- (Includes new option widgets)
download_btn.on_click(on_button_click)

drive_status_html = ""
if not drive_mounted: drive_status_html = "<p style='color: orange; font-weight: bold;'>Warning: Google Drive not mounted...</p>"

gui = widgets.VBox([
    widgets.HTML(value=drive_status_html),
    widgets.HTML("<h2>YouTube Downloader</h2>"),
    widgets.Label("1. Enter URL & Target Folder:"),
    url_widget,
    drive_folder_widget,
    widgets.Label("2. Authentication Cookie File (Required):"),
    cookies_path_widget,
    cookies_help_html,
    widgets.Label("3. Download Options:"),
    options_box, # Box containing new options
    widgets.HTML("<hr>"),
    playlist_range_box, # Initially hidden HBox for playlist range
    download_btn,
    widgets.HTML("<hr>"),
    progress_widget,
    status_widget,
    output_widget
])

logger.info("GUI components created. Displaying UI.")
display(gui)